# Определение математических заблуждений — полный ноутбук

Здесь всё в одном месте: скачивание данных с Kaggle, обучение модели и сохранение файлов для веб-приложения (Hugging Face).

**Важно перед стартом:** зайди на страницу соревнования и нажми **Join / Accept rules**, иначе Kaggle не даст скачать данные:
https://www.kaggle.com/competitions/map-charting-student-math-misunderstandings/rules

Просто запускай ячейки по порядку сверху вниз.

## Шаг 1. Ключ Kaggle API
На kaggle.com: **Account → Settings → Create New API Token** — скачается файл `kaggle.json`. Запусти ячейку и выбери его.

In [ ]:
from google.colab import files
print("Выбери файл kaggle.json:")
files.upload()

# кладём ключ туда, где его ждёт библиотека kaggle
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print("Ключ установлен.")

## Шаг 2. Скачиваем и распаковываем данные

In [ ]:
!pip install kaggle -q
!kaggle competitions download -c map-charting-student-math-misunderstandings
!unzip -o map-charting-student-math-misunderstandings.zip -d data

import os
print("\nФайлы в папке data:")
print(os.listdir("data"))

## Шаг 3. Читаем таблицу

In [ ]:
import pandas as pd

df = pd.read_csv("data/train.csv", on_bad_lines="skip")

print("Размер таблицы:", df.shape)
print("Колонки:", df.columns.tolist())
df.head()

## Шаг 4. Готовим данные
Собираем текст `вопрос + объяснение` — так же, как потом будет в приложении. Цель — колонка `Misconception` (у правильных ответов ошибки нет → `No_Misconception`).

In [ ]:
# при необходимости подгони имена под то, что вывелось в Шаге 3
TEXT_QUESTION = "QuestionText"
TEXT_EXPLAIN  = "StudentExplanation"
TARGET_COL    = "Misconception"

df[TARGET_COL] = df[TARGET_COL].fillna("No_Misconception")
df["text"] = (df[TEXT_QUESTION].fillna("") + " " + df[TEXT_EXPLAIN].fillna("")).str.strip()

X = df["text"]
y = df[TARGET_COL]

print("Всего примеров:", len(df))
print("Классов:", y.nunique())
print("\nСамые частые классы (сверь их с ключами словаря answers):")
print(y.value_counts().head(15))

## Шаг 5. Делим train/test
Один вопрос встречается у многих учеников. Чтобы модель не «подсмотрела» ответ, делим **по вопросам** (`QuestionId`), а не случайно по строкам.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

if "QuestionId" in df.columns:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr, te = next(gss.split(X, y, groups=df["QuestionId"]))
    X_train, X_test = X.iloc[tr], X.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    print("Деление по вопросам (без утечки).")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print("Обычное деление со стратификацией.")

print("Обучение:", len(X_train), "| Тест:", len(X_test))

## Шаг 6. Baseline — «отметка на стене»
Простейшая модель: всегда самый частый класс. Моя модель обязана быть лучше неё.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
bp = baseline.predict(X_test)

print("BASELINE")
print("  Accuracy:", round(accuracy_score(y_test, bp), 3))
print("  macro-F1:", round(f1_score(y_test, bp, average="macro"), 3))

## Шаг 7. Моя модель: TF-IDF + логистическая регрессия (в Pipeline)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
]).fit(X_train, y_train)

pred = model.predict(X_test)

print("МОЯ МОДЕЛЬ")
print("  Accuracy:", round(accuracy_score(y_test, pred), 3))
print("  macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))
print("\n(эти числа впиши в таблицу метрик в README)")
print("\n", classification_report(y_test, pred, zero_division=0))

## Шаг 8. Сохраняем модель и русские ответы → скачиваем
Эти два файла заливаем в Hugging Face Space рядом с `app.py`.

In [ ]:
import joblib
from google.colab import files

answers = {'No_Misconception': 'Ошибок не обнаружено. Объяснение ученика выглядит корректным.', 'Incomplete': 'Объяснение неполное — пропущена важная часть рассуждения.', 'Wrong_fraction': 'Возможна ошибка при работе с дробями.', 'Wrong_Fraction': 'Возможна ошибка при работе с дробями.', 'Wrong_term': 'Возможно, математический термин используется неверно.', 'Additive': 'Возможно, применяется сложение там, где нужна другая операция.', 'Subtraction': 'Возможно, неверно применяется вычитание.', 'Division': 'Возможно, неверно применяется деление.', 'Mult': 'Возможно, неверно применяется умножение.', 'Inversion': 'Возможно, неверно понято обратное действие или преобразование.', 'Duplication': 'Возможно, одно и то же значение учитывается или применяется дважды.', 'Positive': 'Возможно, неверно понят знак или положительное значение.', 'Scale': 'Ошибка может быть связана с масштабом — изменением величины.', 'Whole_numbers_larger': 'Возможно, ученик считает, что большее целое число всегда означает большую величину.', 'Not_variable': 'Возможно, неверно понята роль переменной.', 'Wrong_Operation': 'Возможно, выбрана неверная математическая операция.', 'WNB': 'Рассуждение может быть неверным или недостаточным.', 'Irrelevant': 'В объяснении есть информация, не относящаяся к решению задачи.', 'Unknowable': 'Возможно, ученик считает, что ответ нельзя определить из данных условия.', 'Adding_across': 'Ученик складывает числители и знаменатели вместо приведения к общему знаменателю.'}

joblib.dump(model, "math_misconception_model.pkl")
joblib.dump(answers, "answers.pkl")

files.download("math_misconception_model.pkl")
files.download("answers.pkl")
print("Готово — скачай оба файла и залей в Space.")